In [2]:
import numpy as np
import pandas as pd
from IPython.display import display

In [3]:
df = pd.read_csv("/content/drive/MyDrive/cyclistic_case_study/working_data/cyclistic_ride_13_months_cleaned_data.csv")
print(df.info(show_counts=True))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6219779 entries, 0 to 6219778
Data columns (total 18 columns):
 #   Column               Non-Null Count    Dtype  
---  ------               --------------    -----  
 0   ride_id              6219779 non-null  object 
 1   rideable_type        6219779 non-null  object 
 2   started_at           6219779 non-null  object 
 3   ended_at             6219779 non-null  object 
 4   start_station_name   4942568 non-null  object 
 5   start_station_id     4942568 non-null  object 
 6   end_station_name     4901582 non-null  object 
 7   end_station_id       4901582 non-null  object 
 8   start_lat            6219779 non-null  float64
 9   start_lng            6219779 non-null  float64
 10  end_lat              6219779 non-null  float64
 11  end_lng              6219779 non-null  float64
 12  member_casual        6219779 non-null  object 
 13  station_name_status  6219779 non-null  object 
 14  ride_length          6219779 non-null  float64
 15

In [4]:
# mean of ride length
mean_ride_length = round(df["ride_length"].mean())
print(f"Average ride length: {mean_ride_length}")

#max ride length
max_ride_length = df["ride_length"].max()
print(f"Maximum ride length (in min): {max_ride_length}")

#calculating mode of day of week
day_mode = df["day_of_week"].mode()[0]
print(f"Most popular day for ride: {day_mode}")

Average ride length: 15
Maximum ride length (in min): 1500.0
Most popular day for ride: Saturday


In [5]:
# statistics by member type
df.groupby("member_casual").agg(
    avg_duration = ("ride_length", "mean"),
    num_of_rides = ("ride_id", "count"),
    common_month = ("months", lambda z: z.mode().iloc[0]),
    common_day = ("day_of_week", lambda x: x.mode().iloc[0]),
    common_hour = ("hours", lambda y: y.mode().iloc[0]),

)


,avg_duration,num_of_rides,common_month,common_day,common_hour
member_casual,,,,,
casual,19.564093,2214056,May,Saturday,17
member,12.231712,4005723,May,Thursday,17


In [6]:
# putting days in order
days_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
df['day_of_week'] = pd.Categorical(df["day_of_week"], categories=days_order, ordered=True)

#putting months in order
months_order = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]


# Ride data by member type and weekday with sorted days
# The number of rides by rider type & weekday and the average ride duration by rider type and weekday
rider_weekday_stat=  df.groupby(["member_casual", "day_of_week"], observed=True).agg(
    num_of_rides = ("ride_id", "count"),
    avg_duration = ("ride_length", "mean"),
  )



# Ride data by member type and months with sorted months
rider_months_stat = df.groupby(["member_casual", "months"], observed=True).agg(
    num_of_rides = ("ride_id", "count"),
    avg_duration = ("ride_length", "mean")
)


display(rider_weekday_stat)
print("\n \n")
display(rider_months_stat)


num_of_rides  avg_duration
member_casual day_of_week                            
casual        Monday             257651     19.735161
              Tuesday            245891     17.062589
              Wednesday          246468     16.234797
              Thursday           284450     17.107636
              Friday             346319     19.073493
              Saturday           459661     21.947696
              Sunday             373616     22.681143
member        Monday             557585     11.980181
              Tuesday            628669     11.809533
              Wednesday          629826     11.706460
              Thursday           650242     11.768137
              Friday             595813     12.124598
              Saturday           506967     13.385927
              Sunday             436621     13.414856

num_of_rides  avg_duration
member_casual months                               
casual        April            128289     16.682989
              August           327253     21.558788
              December          27381     12.800482
              February          40342     14.077909
              January           24119     12.890004
              July             312344     21.235750
              June             282151     21.728018
              March             85651     16.805221
              May              415948     20.052259
              November          95964     15.029845
              October          217121     17.980232
              September        257493     19.423938
member        April            310677     11.470308
              August           445629     13.013594
              December         110090     11.665174
              February         158131     10.705402
              January          110705     11.594273
              July             433256     13.114644
              June             381120     12.797171
              March            225349     11.279083
              May              717485     12.146491
              November         253578     11.649141
              October          416439     12.095930
              September        443264     12.557620

In [7]:
# distribution by bike type

df.groupby(["rideable_type", "member_casual"]).agg(
    num_of_rides = ("ride_id", "count")
)

num_of_rides
rideable_type member_casual              
classic_bike  casual               724041
              member              1401774
electric_bike casual              1490015
              member              2603949

In [8]:
#checking if there is reasion behind many missing station_name_status

pd.crosstab(df['rideable_type'], df['station_name_status'])

# electric bikes can often be parked in random places because of built in locks

station_name_status,Complete,Missing
rideable_type,,
classic_bike,2125733,82
electric_bike,2066977,2026987
